# R15 — 학습된 판정기로 재채점 (재학습 없음)

사전등록 `R15_R17_CAMERA_READY.md` §R15. 저장된 completion만 읽는다 — 생성·학습 없음.
판정기: `meta-llama/Llama-Guard-3-8B`, `cais/HarmBench-Llama-2-13b-cls` (4-bit).
입력: 헤드라인 `_env76` 12파일(15,024) + 이전 seed-42 grid 4파일(5,008, 인간 검증 표본의 출처).

**필요한 것**: GPU T4 x2 · Secrets `HF_TOKEN`(Llama-Guard-3 게이트 승인 계정) ·
Input `branch-1-premise`(새 버전), `gen-data-backup`.

★이 파일을 직접 수정하지 말 것 — `src/make_nb_camera_ready.py`가 생성한다.

## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
비교 가능성 핀: `unsloth==2026.7.6`, `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth==2026.7.6")
pip("transformers==4.56.2", "trl==0.22.2")

from importlib.metadata import version
_installed_unsloth = version("unsloth")
assert _installed_unsloth == "2026.7.6", (
    f"Unsloth 설치 핀 불일치: 기대=2026.7.6 실제={_installed_unsloth}")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

In [ ]:
import subprocess
def run(cmd):
    print(f"\n$ {cmd}", flush=True)
    assert subprocess.run(cmd, shell=True).returncode == 0, f"실패: {cmd}"


In [ ]:
import json, os
NB_NAME = "nb_r15_judges"
NB_REV = "R15a"
DATASET_OK = ['R15a']
_rev = json.load(open("configs/arms.json", encoding="utf-8")).get("nb_revisions", {}).get(NB_NAME)
assert _rev in DATASET_OK, (f"리비전 불일치: 노트북={NB_NAME}/{NB_REV} Dataset={_rev} (허용 {DATASET_OK}) — "
                            "Dataset을 New Version으로 다시 올리거나 노트북을 다시 Import하라")
assert os.path.exists("R15_R17_CAMERA_READY.md"), "사전등록 문서가 Dataset에 없다 — 커밋·업로드 후 실행"
print(f"[gate] {NB_NAME}/{NB_REV} (Dataset {_rev}) · 사전등록 문서 확인")


## 재채점

In [ ]:
import glob, os, subprocess, sys
pip("bitsandbytes>=0.43", "accelerate")

HEADLINE = sorted(glob.glob("results/eval_arm*_r*_s*_env76.json"))
LEGACY = [f"results/eval_arm0_r0_s42.json"] + [f"results/eval_arm{a}_r3_s42.json" for a in (1, 2, 3)]
assert len(HEADLINE) == 12, f"헤드라인 eval 파일이 12개가 아니다: {len(HEADLINE)}"
assert all(os.path.exists(p) for p in LEGACY), f"이전 grid 파일 누락: {LEGACY}"
run("python src/rescore_judges.py --selftest")

# ★ 커밋마다 이 값 하나만 바꾼다. 1 = 두 판정기 모두(기본, ~3-4h) · 2 = Llama-Guard만 · 3 = HarmBench만
BATCH = 1
JUDGE_SETS = {1: ["llamaguard3", "harmbench"], 2: ["llamaguard3"], 3: ["harmbench"]}
for judge in JUDGE_SETS[BATCH]:
    bs = 8 if judge == "llamaguard3" else 4
    files = " ".join(f'--files "{p}"' for p in HEADLINE + LEGACY)
    run(f"python src/rescore_judges.py --judge {judge} --batch_size {bs} {files}")
    import shutil
    shutil.copy(f"results/judge_{judge}.json", f"/kaggle/working/judge_{judge}.json")
    print(f"[snap] /kaggle/working/judge_{judge}.json", flush=True)


## 결과 회수

In [ ]:
import shutil, os, glob
zp = shutil.make_archive(f"/kaggle/working/r15_results_batch{BATCH}", "zip",
                         "/kaggle/working/experiment/results")
print("zip:", zp, "| MB:", round(os.path.getsize(zp) / 1e6, 2))
for p in sorted(glob.glob("/kaggle/working/experiment/results/*judge_*")):
    print(f"  {os.path.getsize(p)/1024:8.0f} KB  {os.path.basename(p)}")
